# LED OCR ablation, phase 3: generate and score (Colab T4)
Protocol `led-ocr-ablation-261110-v1`. Same paths and SET as the extract notebook. Summary text stays private; no comparison is computed here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, subprocess
from pathlib import Path

APP = '/content/multimodal-lecture-summarizer'   # app repo (separate repo)
BENCH = '/content/benchmark'                     # benchmark repo (separate repo, private)
WORK = '/content/drive/MyDrive/led-ocr-ablation-261110'   # private: videos/, run/
SET = 'smoke'                                    # 'smoke' first, then 'cohort'
APP_COMMIT = '5e26ea799de9ac7e5a934d684ed4a031ff1202e4'  # frozen in protocol.json
PY = '/content/venv/bin/python'                  # same layout as the app's Colab worker notebook
PROTOCOL = f'{BENCH}/experiments/registry/protocols/led-ocr-ablation-261110-v1'

# benchmark repo is private: add a Colab secret GITHUB_TOKEN (read access) and enable notebook access.
try:
    from google.colab import userdata
    _tok = userdata.get('GITHUB_TOKEN')
except Exception:
    _tok = ''

def _clone(repo, dest):
    if os.path.isdir(dest): return
    base = f'x-access-token:{_tok}@' if _tok else ''
    r = subprocess.run(['git', 'clone', '-q', f'https://{base}github.com/multimodal-lecture-summarizer/{repo}.git', dest],
                       env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    assert r.returncode == 0, f'clone of {repo} failed (private repo? set the GITHUB_TOKEN Colab secret)'

_clone('benchmark', BENCH); _clone('multimodal-lecture-summarizer', APP)
subprocess.run(['git', '-C', APP, 'checkout', '-q', APP_COMMIT], check=True)  # the scripts refuse any other app commit
os.makedirs(f'{WORK}/videos', exist_ok=True); os.makedirs(f'{WORK}/run', exist_ok=True)

hashes = json.load(open(f'{PROTOCOL}/hashes.json'))['sha256']
for name, expected in hashes.items():
    assert hashlib.sha256(open(f'{PROTOCOL}/{name}', 'rb').read()).hexdigest() == expected, name
print('frozen files verified')

In [ ]:
# Python 3.11 venv as in the app's Colab worker; Colab's own Python has no torch 2.5.1 wheels.
if not Path(PY).is_file():
    subprocess.run(['uv', 'venv', '--python', '3.11', '/content/venv'], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', PY, '-r', f'{APP}/ai_workers/requirements.txt',
                '-r', f'{APP}/backend/requirements.txt', 'setuptools'], check=True, cwd=APP)
subprocess.run(['uv', 'pip', 'install', '--python', PY, '--no-deps', '-r', f'{APP}/ai_workers/requirements-tracking.txt'], check=True, cwd=APP)
subprocess.run(['nvidia-smi', '-L'])

In [ ]:
site = '/content/venv/lib/python3.11/site-packages'
ENV = {**os.environ,
       'LD_LIBRARY_PATH': f'{site}/nvidia/cudnn/lib:{site}/nvidia/cublas/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
       'PATH': '/content/venv/bin:' + os.environ['PATH'],
       'PADDLEOCR_USE_GPU': '0', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8',
       'PROCESS_MIN_AVAILABLE_MEMORY_MB': '4096', 'PROCESS_SOFT_MIN_AVAILABLE_MEMORY_MB': '3072',
       'HF_HOME': '/content/hf'}

import time, datetime
LOG_DIR = Path(f'{WORK}/run/logs'); LOG_DIR.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, env=None, tag='step'):
    """Stream output to the cell, tee it to a Drive log file, print timing, stop on a non-zero exit."""
    logfile = LOG_DIR / f'{SET}-{tag}-{datetime.datetime.now():%Y%m%d-%H%M%S}.log'
    print(f'$ {" ".join(map(str, cmd))}\n  cwd={cwd}  log={logfile}', flush=True)
    started = time.time()
    p = subprocess.Popen(cmd, cwd=cwd, env=env or ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with open(logfile, 'w', encoding='utf-8') as out:
        for line in p.stdout:
            print(line, end=''); out.write(line); out.flush()
    code = p.wait()
    print(f'\n[{tag}] exit code {code} after {time.time() - started:.0f}s; full log: {logfile}')
    assert code == 0, f'{tag} failed with exit code {code}; read the output above (tracebacks also in {LOG_DIR}/errors.log)'

def show_errors(n=60):
    f = LOG_DIR / 'errors.log'
    print(''.join(f.read_text(encoding='utf-8').splitlines(True)[-n:]) if f.exists() else 'no errors.log (no failed video/cell)')

In [ ]:
# 3a. Generate A then D per video in the app venv. Resumable; an error cell is final.
run([PY, '-m', 'experiments.scripts.led_ocr_ablation_generate', '--app-root', APP, '--out-dir', f'{WORK}/run',
     '--set', SET, '--protocol-dir', PROTOCOL], cwd=BENCH, tag='generate')
show_errors()

In [ ]:
# Audit before scoring: both arms present, same input hash, arm A source is the plain transcript.
import sys; sys.path.insert(0, BENCH)
from experiments.scripts.led_ocr_ablation_generate import source_check
from experiments.scripts.led_ocr_ablation_extract import read_rows
cells = read_rows(Path(f'{WORK}/run/cells_receipt_{SET}.jsonl'))
ok = [c for c in cells if c.get('status') == 'ok']
print(len(cells), 'cells,', len(cells) - len(ok), 'error cells')
print(source_check(ok) or 'source audit clean')
print('D cells with no OCR in the input:', [c['id'] for c in ok if c['arm'] == 'D' and c['ocr_words'] == 0])

In [ ]:
# 3b. Scoring uses Colab's own Python (torch/transformers already there), not the app venv.
# Only the scorer's libraries are installed; benchmark/requirements.txt is for other experiments.
run(['pip', '-q', 'install', 'rouge-score==0.1.2', 'bert-score==0.3.13', 'nltk', 'datasets', 'sentence-transformers'], env=dict(os.environ), tag='score-install')
run(['pip', '-q', 'install', 'minicheck @ git+https://github.com/Liyan06/MiniCheck.git@b58b9fa69acbd1015ec970fa65dd752413a053d2'], env=dict(os.environ), tag='score-install')

In [ ]:
# Prints no comparison. References come from gigant/tib (pinned revision) unless --references <private json> is given.
run(['python', '-m', 'experiments.scripts.led_ocr_ablation_score', '--out-dir', f'{WORK}/run', '--set', SET], cwd=BENCH, env=dict(os.environ), tag='score')

In [ ]:
# Compact per-cell table (text-free): generation receipt joined with scores. Paste this output when reporting.
import pandas as pd
cells = pd.DataFrame(read_rows(Path(f'{WORK}/run/cells_receipt_{SET}.jsonl')))
scores = pd.DataFrame(read_rows(Path(f'{WORK}/run/scores_receipt_{SET}.jsonl')))
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
keep = [c for c in ['id','stratum','arm','status','error','detail','model_used','word_count','led_calls','transcript_words','ocr_words','slides_used','source_observed_match','generation_seconds'] if c in cells]
print(cells[keep].to_string(index=False))
if len(scores):
    flat = scores.copy()
    for col in ['minicheck_evidence_full', 'minicheck_evidence_transcript']:
        flat[col + '_mean'] = flat[col].map(lambda v: v.get('mean_prob') if isinstance(v, dict) else None)
        flat[col + '_flag'] = flat[col].map(lambda v: v.get('flagged_share') if isinstance(v, dict) else None)
    cols = ['id','arm','word_count','rougeL_f1_at_350','rougeL_f1_at_120','bertscore_f1','minicheck_evidence_full_mean','minicheck_evidence_full_flag','minicheck_evidence_transcript_mean','slide_term_coverage','errors']
    print(flat[[c for c in cols if c in flat]].round(4).to_string(index=False))